# 01 · Limpieza y organización del repositorio

**Objetivo:** documentar el proceso de limpieza inicial del repositorio antes de comenzar la fase de desarrollo basada en notebooks.

**Contexto:** durante las primeras semanas de desarrollo del TFG se trabajó con scripts Python sueltos y se subieron al repositorio archivos pesados que no deberían estar versionados (vídeos del dataset, frames extraídos, modelos entrenados, configuración del IDE). Este notebook deja constancia del proceso de limpieza y reorganización del repositorio hacia la estructura definitiva basada en notebooks.

**Salida esperada:** repositorio limpio, con `.gitignore` correcto y archivos pesados eliminados del control de versiones manteniendo los datos en local.

## 1. Diagnóstico inicial

Antes de borrar nada, conviene listar qué archivos están actualmente versionados en el repositorio para identificar cuáles deben mantenerse y cuáles eliminarse del control de versiones.

In [1]:
import os
import subprocess
from collections import Counter
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
os.chdir(REPO_ROOT)
print(f"Directorio de trabajo: {os.getcwd()}")

Directorio de trabajo: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG


In [2]:
result = subprocess.run(["git", "ls-files"], capture_output=True, text=True)
files = result.stdout.strip().split("\n")

print(f"Total de archivos versionados: {len(files)}\n")
print("Distribución por carpeta:")

folders = Counter()
for f in files:
    top = f.split("/")[0] if "/" in f else "(raíz)"
    folders[top] += 1

for folder, count in sorted(folders.items(), key=lambda x: -x[1]):
    print(f"  {folder}: {count} archivos")

Total de archivos versionados: 1111

Distribución por carpeta:
  data: 1077 archivos
  runs: 22 archivos
  .idea: 6 archivos
  (raíz): 5 archivos
  src: 1 archivos


## 2. Identificación de archivos a eliminar del control de versiones

Los archivos siguientes están versionados pero no deberían estarlo:

| Carpeta o archivo | Motivo de exclusión |
|---|---|
| `data/raw/` | Vídeos del dataset DFL Bundesliga (Kaggle), externos al proyecto |
| `data/frames/` | Frames extraídos, regenerables ejecutando el notebook de extracción |
| `data/dataset/` | Dataset descargado de Roboflow, regenerable bajo demanda |
| `data/auto_labeled/` | Resultados intermedios del auto-etiquetado |
| `runs/` | Resultados de entrenamientos de YOLO, regenerables |
| `models/*.pt` | Modelos entrenados, ya alojados en Hugging Face Hub |
| `*.pt` en raíz | Pesos pre-entrenados de Ultralytics, se descargan automáticamente |
| `.idea/` | Configuración personal del IDE PyCharm |

La justificación común es que estos archivos son **datos** o **artefactos generados**, no **código fuente**. El repositorio debe contener únicamente el código que permite regenerar estos artefactos a partir de las fuentes de datos correspondientes.

## 3. Creación del `.gitignore`

El archivo `.gitignore` indica a Git qué archivos o carpetas debe ignorar permanentemente. Es la primera línea de defensa para que los archivos pesados no vuelvan a versionarse en futuros commits.

In [3]:
gitignore_content = """# Datos pesados
data/raw/
data/frames/
data/dataset/
data/auto_labeled/

# Modelos y resultados de entrenamiento
runs/
models/
*.pt
*.cache

# Salidas del pipeline
outputs/

# Python
__pycache__/
*.pyc
*.pyo
.env
.venv/
venv/

# Jupyter
.ipynb_checkpoints/

# IDE
.idea/
.vscode/

# Sistema
.DS_Store
Thumbs.db
"""

with open(".gitignore", "w", encoding="utf-8") as f:
    f.write(gitignore_content)

print(".gitignore creado correctamente")
print(f"Tamaño: {len(gitignore_content)} caracteres")

.gitignore creado correctamente
Tamaño: 311 caracteres


## 4. Eliminación del control de versiones

El comando `git rm --cached` elimina los archivos del seguimiento de Git pero los mantiene en disco. Esto significa que los datos siguen disponibles localmente para trabajar, pero ya no se subirán al repositorio remoto.

Importante: el modelo `best.pt` ya fue subido previamente a Hugging Face Hub en el notebook anterior, por lo que puede eliminarse del control de versiones sin riesgo de pérdida.

In [4]:
rutas_a_eliminar = [
    ("-r", "data/raw"),
    ("-r", "data/frames"),
    ("-r", "data/dataset"),
    ("-r", "data/auto_labeled"),
    ("-r", "runs"),
    ("-r", "models"),
    ("-r", ".idea"),
    ("", "yolov8s.pt"),
    ("", "yolo26n.pt"),
    ("", "src/data/extract_frames.py"),
    ("", "train.py"),
    ("", "tfg.ipynb"),
]

for flag, ruta in rutas_a_eliminar:
    cmd = ["git", "rm"]
    if flag:
        cmd.append(flag)
    cmd.extend(["--cached", ruta])

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0:
        n_eliminados = len(result.stdout.strip().split("\n")) if result.stdout.strip() else 0
        print(f"OK    · {ruta} ({n_eliminados} archivos)")
    else:
        print(f"SKIP  · {ruta} (no existe en el repo o ya estaba fuera)")

OK    · data/raw (12 archivos)
OK    · data/frames (756 archivos)
OK    · data/dataset (309 archivos)
SKIP  · data/auto_labeled (no existe en el repo o ya estaba fuera)
OK    · runs (22 archivos)
SKIP  · models (no existe en el repo o ya estaba fuera)
OK    · .idea (6 archivos)
OK    · yolov8s.pt (1 archivos)
OK    · yolo26n.pt (1 archivos)
OK    · src/data/extract_frames.py (1 archivos)
OK    · train.py (1 archivos)
OK    · tfg.ipynb (1 archivos)


## 5. Verificación del estado del repositorio tras la limpieza

In [5]:
result = subprocess.run(["git", "ls-files"], capture_output=True, text=True)
files = result.stdout.strip().split("\n") if result.stdout.strip() else []

print(f"Archivos versionados tras la limpieza: {len(files)}\n")
print("Distribución por carpeta:")

folders = Counter()
for f in files:
    top = f.split("/")[0] if "/" in f else "(raíz)"
    folders[top] += 1

for folder, count in sorted(folders.items(), key=lambda x: -x[1]):
    print(f"  {folder}: {count} archivos")

Archivos versionados tras la limpieza: 1

Distribución por carpeta:
  (raíz): 1 archivos


## 6. Creación de la estructura de carpetas para los notebooks

Una vez limpio el repositorio, se crea la estructura de carpetas que organizará los notebooks por fases del proyecto.

In [6]:
carpetas_notebooks = [
    "notebooks/fase1_preparacion",
    "notebooks/fase2_deteccion_jugadores",
    "notebooks/fase3_keypoints_homografia",
    "notebooks/fase4_tracking_equipos",
    "notebooks/fase5_homografia_pizarra",
    "notebooks/fase6_motor_tactico",
    "notebooks/fase7_visualizacion",
    "notebooks/fase8_evaluacion",
]

for carpeta in carpetas_notebooks:
    os.makedirs(carpeta, exist_ok=True)
    placeholder = os.path.join(carpeta, ".gitkeep")
    if not os.path.exists(placeholder):
        with open(placeholder, "w") as f:
            pass
    print(f"OK · {carpeta}")

OK · notebooks/fase1_preparacion
OK · notebooks/fase2_deteccion_jugadores
OK · notebooks/fase3_keypoints_homografia
OK · notebooks/fase4_tracking_equipos
OK · notebooks/fase5_homografia_pizarra
OK · notebooks/fase6_motor_tactico
OK · notebooks/fase7_visualizacion
OK · notebooks/fase8_evaluacion


## 7. Preparación del commit

Se preparan los cambios para el commit, dejando una traza clara del proceso de limpieza en el historial del repositorio.

In [7]:
subprocess.run(["git", "add", ".gitignore"], check=True)
subprocess.run(["git", "add", "notebooks/"], check=True)

result = subprocess.run(["git", "status", "--short"], capture_output=True, text=True)
print("Estado de Git tras la preparación del commit:\n")
print(result.stdout)

Estado de Git tras la preparación del commit:

A  .gitignore
D  .idea/.gitignore
D  .idea/TFG.iml
D  .idea/inspectionProfiles/profiles_settings.xml
D  .idea/misc.xml
D  .idea/modules.xml
D  .idea/vcs.xml
D  data/dataset/README.dataset.txt
D  data/dataset/README.roboflow.txt
D  data/dataset/data.yaml
D  data/dataset/train/images/A1606b0e6_0-12-_frame00000_jpg.rf.a1cb29c99f3d4bea5bf22144777bde60.jpg
D  data/dataset/train/images/A1606b0e6_0-12-_frame00012_jpg.rf.6cb65a91ec05e815172c285b5c241722.jpg
D  data/dataset/train/images/A1606b0e6_0-12-_frame00024_jpg.rf.9113978a860e7cd67c9f658f157177e9.jpg
D  data/dataset/train/images/A1606b0e6_0-12-_frame00036_jpg.rf.972c50e482a3f9e40ec63982487066db.jpg
D  data/dataset/train/images/A1606b0e6_0-12-_frame00048_jpg.rf.8f028c2408809e7e27f7c7182c05b671.jpg
D  data/dataset/train/images/A1606b0e6_0-12-_frame00084_jpg.rf.d2d3905c6adf50a5930a19b8d1e42381.jpg
D  data/dataset/train/images/A1606b0e6_0-12-_frame00096_jpg.rf.272ad9e40ab8952e1ac609b33566d7e3.jpg

## 8. Conclusiones

Tras este notebook:

- El repositorio queda limpio de archivos pesados (vídeos, frames, modelos, resultados de entrenamiento).
- El `.gitignore` evita que estos archivos vuelvan a versionarse en futuros commits.
- La estructura de carpetas por fases queda preparada para alojar los notebooks de cada parte del proyecto.
- Los datos siguen disponibles en local para trabajar, pero ya no se versionan en GitHub.
- El modelo entrenado está alojado de forma segura en Hugging Face Hub, garantizando la reproducibilidad del trabajo sin sobrecargar el repositorio.

A partir de este punto, todos los desarrollos del proyecto se documentarán en notebooks organizados por fases.